# Setup Kaggle with VS Code Remote Tunnels

This notebook starts a **VS Code Remote Tunnel** so you can connect from your local VS Code.
No SSH keys, no ngrok, no credit card needed.

## Quick Start
1. **Internet must be ON** (Session options gear icon)
2. Run **Cell 1** (bootstrap CLI â€” one-time per environment)
3. Run **Cell 2** (start tunnel) â€” authorize with GitHub when prompted
4. In local VS Code: Remote Explorer â†’ Connect to Tunnel â†’ pick `kaggle-gpu`
5. Open `/kaggle/working` and clone/upload your code

##### Cells 1 & 2 are executable on Kaggle.

In [ ]:
# Download VS Code CLI (one-time per clean environment)
# If /kaggle/working is persisted and CLI is already downloaded, skip this.

import urllib.request, os, tarfile, glob

extract_dir = "/kaggle/working/vscode-cli"
if glob.glob(f"{extract_dir}/**/code", recursive=True):
    print("VS Code CLI already downloaded â€” skipping")
else:
    url = "https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64"
    tar_path = "/kaggle/working/vscode-cli.tar.gz"

    urllib.request.urlretrieve(url, tar_path)
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(tar_path) as tar:
        tar.extractall(extract_dir)
    os.remove(tar_path)

    code_paths = glob.glob(f"{extract_dir}/**/code", recursive=True)
    if code_paths:
        os.chmod(code_paths[0], 0o755)
        print(f"VS Code CLI ready: {code_paths[0]}")
    else:
        print("Contents:", os.listdir(extract_dir))
        raise FileNotFoundError("code binary not found after extraction")

In [ ]:
# Start VS Code Remote Tunnel
# This cell blocks â€” keep it running while you use the tunnel.
# Interrupt or stop this cell to terminate the connection.

import subprocess, os, glob

extract_dir = "/kaggle/working/vscode-cli"
code_paths = glob.glob(f"{extract_dir}/**/code", recursive=True)
code_path = code_paths[0]
os.environ["PATH"] = os.path.dirname(code_path) + os.pathsep + os.environ.get("PATH", "")

tunnel_name = "kaggle-gpu"
cmd = [
    code_path, "tunnel", "--accept-server-license-terms",
    "--name", tunnel_name, "--verbose"
]

print(f"Starting tunnel '{tunnel_name}'...")
print("Check the output below for a device-login URL.")
print("=" * 60)

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

for line in iter(process.stdout.readline, ""):
    print(line, end="")
    if "To grant access" in line or "device" in line.lower():
        print()
        print("=" * 60)
        print("  STEP: Open the URL above, enter the code, authorize with GitHub")
        print("=" * 60)
        print()
        print("  Then in local VS Code:")
        print("    1. Remote Explorer -> Connect to Tunnel")
        print(f"    2. Sign in with same GitHub account")
        print(f"    3. Select tunnel '{tunnel_name}'")
        print("    4. Open folder /kaggle/working")
        print()

print("Tunnel closed.")

##### Run these using the kaggle runtime provided by the tunnel

In [ ]:
# Clone project (or upload through VS Code)
# Replace with your repo URL

import os
# if not os.path.exists("/kaggle/working/medical_CTTA"):
#     !cd /kaggle/working && git clone https://github.com/<username>/medical_CTTA.git

from src.env import init
init()
!pip install -e .
print("Project installed")

In [1]:
# Download datasets

import os, sys
_d = os.getcwd()
PROJECT_ROOT = None
while _d != os.path.dirname(_d):
    if os.path.exists(os.path.join(_d, 'setup.py')):
        PROJECT_ROOT = _d
        break
    _d = os.path.dirname(_d)
if PROJECT_ROOT is None:
    for _p in ['/kaggle/working/medical_CTTA', '/content/medical_CTTA']:
        if os.path.isdir(_p):
            PROJECT_ROOT = _p
            break
if PROJECT_ROOT and PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.env import init
init()
from src.data.download import DatasetDownloader

downloader = DatasetDownloader(data_dir=os.path.join(PROJECT_ROOT, 'data'))
existing = downloader.verify_datasets()

all_exist = all(existing.get(d, {}).get('exists') for d in ['idrid', 'aptos2019', 'messidor2'])
if all_exist:
    print('All datasets already downloaded -- skipping.')
else:
    print('Downloading datasets from Kaggle ...')
    if not existing.get('idrid', {}).get('exists'):
        downloader.download_idrid()
    if not existing.get('aptos2019', {}).get('exists'):
        downloader.download_aptos2019()
    if not existing.get('messidor2', {}).get('exists'):
        downloader.download_messidor2()

for name, info in downloader.verify_datasets().items():
    ok = 'OK' if info.get('exists') else 'MISSING'
    print(f'  [{ok}] {name}')
print('Done!')

[env] Dataset symlink: /kaggle/working/medical_CTTA/data -> /kaggle/working/data
  [OK] idrid
  [OK] aptos2019
  [OK] messidor2
Done!


In [2]:
# Verify GPU + setup
import torch
from src.config import load_config
from src.models.registry import ModelRegistry
from src.data.registry import DatasetRegistry
from src.adapters.registry import AdapterRegistry

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

config = load_config('configs/default.yaml')
print(f'Models:   {ModelRegistry.list_models()}')
print(f'Datasets: {DatasetRegistry.list_datasets()}')
print(f'Adapters: {AdapterRegistry.list_adapters()}')
print()
print('Setup complete! Ready to run experiments.')

CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB
Models:   ['retfound', 'visionfm']
Datasets: ['idrid', 'aptos2019', 'messidor2']
Adapters: ['cotta', 'palm', 'vida', 'ecotta', 'lcotta']

Setup complete! Ready to run experiments.
